# RoSE + query decomposition — ChartQAPro factoid (first 250 questions)

Two decomposition algorithms, evaluated on the **same first 250 factoid
questions in dataset order** (`factoid-00000` …) as the routed run, so
`results/compare_arms.py` can put all arms in one table afterwards.
There is **no routing** in either arm — the final answer always comes
from the full, unchanged RoSE pipeline on **Qwen2.5-VL-7B**.

**Algorithm 1 — decompose → parallel answer → evidence → RoSE.** The 7B
splits the question into two sub-queries (JSON; `[question, question]`
fallback on parse failure). Gemma 3 4B answers sub-query 1 and Qwen3.5-4B
answers sub-query 2, each with 3 sampled paths and the agreement ratio as
confidence. The sub-answers are prepended to the RoSE reasoning prompt as
an evidence block; RoSE's own confidence-weighted vote decides.

**Algorithm 2 — both small models decompose → vote → RoSE.** Gemma and
Qwen3.5 each propose two sub-queries; an `all-mpnet-base-v2` vote merges
agreeing proposals (pairwise cosine ≥ 0.8 after best matching) or picks
the proposal closer to the original question (the rule that fired is
logged per question). RoSE runs with the chosen sub-queries inserted as
"answer these in order, then the main question" steps. No small model
answers anything.

**Runtime + quota.** All three 4-bit VLMs stay resident (~13 GB — use
the **T4 x2** accelerator). Per question: Alg 1 ≈ 7–8 min (7B decompose
+ 2×3 small-model paths + full RoSE on the 7B), Alg 2 ≈ 5 min (2 small
decomposes + full RoSE). At N=250 that is roughly **29–33 h for Alg 1**
and **~21 h for Alg 2** — several chained sessions. With
`SESSION_BUDGET_H = 10.0` every **Save & Run All** version stops cleanly
inside the 12 h cap and PUBLISHES its checkpoints; Algorithm 2 only
starts once Algorithm 1 has finished, so run order is deterministic.
Check `"complete"` in each `decomp_alg<A>_meta.json` and chain sessions
via `RESUME_FROM` (last cell) until both are `true`.


In [ ]:
# ── config ───────────────────────────────────────────────────────
DATASET_KEY = "chartqapro"
ALGORITHMS  = [1, 2]      # run order; Alg 2 starts only if budget remains
N_QUESTIONS = 250         # first N factoid questions in dataset order
DRY_RUN     = False       # True = answer first 2 questions per algorithm
QUANT       = "nf4"       # same quantization as the pilot + routed runs
SESSION_BUDGET_H = 10.0   # stop cleanly after this many hours so a
                          # "Save & Run All" version FINISHES inside the
                          # T4 x2's 12 h session cap and publishes the
                          # checkpoints; None = no budget

REPO_URL  = "https://github.com/ktahsinr/RoSE.git"
REPO_REF  = "feature/decompose-vote"

# Chaining sessions: download decomp_alg<A>_checkpoint.json from a
# finished version's Output pane, attach it (Add Input → Your Work →
# this notebook, or upload as a dataset) and point at it here — answered
# questions are skipped, errored ones retried.
RESUME_FROM = {1: None, 2: None}

import os, time
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # Kaggle can't render the widgets
T_START = time.time()                              # the session budget clock
from pathlib import Path
import sys
try:
    sys.stdout.reconfigure(line_buffering=True)    # live logs in Kaggle's pane
except (AttributeError, ValueError):
    pass

ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"
RESULTS_DIR = WORK / "results" / "decompose"       # published with the notebook
DATASET_JSON = SCRATCH / "chartqapro_factoid_decomp.json"

for d in (SCRATCH, IMAGES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"dataset={DATASET_KEY}  n={N_QUESTIONS}  algorithms={ALGORITHMS}  "
      f"dry_run={DRY_RUN}  quant={QUANT}")


In [ ]:
# ── code + pinned libraries ──────────────────────────────────────
import subprocess
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    REPO_URL, str(CODE_DIR)], check=True)
else:
    # a kernel restart keeps /kaggle/temp — refresh so fixes pushed to the
    # branch mid-session are picked up instead of running stale code
    subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1",
                    "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "reset", "--hard",
                    "FETCH_HEAD"], check=True)
print("repo @", subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(CODE_DIR / "requirements-pinned.txt")], check=True)
# NOTE: transformers==4.51.3 (the sweep pin) supports Gemma 3. If
# Qwen/Qwen3.5-4B fails to load with an unknown-architecture error, install a
# newer transformers here for that session — the run's *_meta.json records the
# exact version either way, so the report stays honest.

sys.path.insert(0, str(CODE_DIR / "chartqapro"))
import transformers
print("transformers:", transformers.__version__)


In [ ]:
# ── HuggingFace auth + gated-access check (all three models) ─────
# Both algorithms touch all three models every question, so access to
# every one is verified up front — better a failure here than hours in.
# check_gated_access DOWNLOADS each config.json, the exact request the
# model load makes (a gated repo's metadata is public, so model_info()
# "succeeds" with no token and the run still 401s at load time).
NO_TOKEN_HELP = (
    "→ No HF token reached this session:\n"
    "  1. hf.co → Settings → Access Tokens → create a READ token\n"
    "     (use the SAME account that accepted the Gemma 3 license)\n"
    "  2. Kaggle → Add-ons → Secrets → add it with label HF_TOKEN\n"
    "  3. TICK THE CHECKBOX attaching the secret to THIS notebook —\n"
    "     the attachment is per-notebook and defaults to off\n"
    "  then re-run this cell.\n")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"could not read Kaggle secret HF_TOKEN: {e}")

if token:
    from huggingface_hub import login
    login(token=token)
    print("HF login ok")
elif ON_KAGGLE:
    raise SystemExit("\nNO HF TOKEN.\n" + NO_TOKEN_HELP)

from decompose.models import check_gated_access
check_gated_access()


In [ ]:
# ── build the first-N ChartQAPro factoid set ─────────────────────
# Ids are factoid ordinals in split order (factoid-00000 …), the repo's
# convention — the SAME window the routed run used, so compare_arms.py
# lines the arms up by id afterwards. No sampling; only these questions
# get their images decoded.
import io, json, ast
from PIL import Image
from datasets import load_dataset

def _gold(v):
    if isinstance(v, (list, tuple)):
        return [str(x) for x in v]
    s = str(v).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                return [str(x) for x in p]
        except Exception:
            pass
    return s

def _save_image(field, path):
    if isinstance(field, dict) and field.get("bytes"):
        img = Image.open(io.BytesIO(field["bytes"]))
    elif isinstance(field, (bytes, bytearray)):
        img = Image.open(io.BytesIO(field))
    elif isinstance(field, Image.Image):
        img = field
    else:
        return str(field)
    img.convert("RGB").save(path, format="PNG")
    return str(path)

ds = load_dataset("ahmed-masry/ChartQAPro", split="test")
qtypes = [str(t).strip().lower() for t in ds["Question Type"]]
fact_idx = [i for i, t in enumerate(qtypes) if t == "factoid"]
print(f"factoid questions in split: {len(fact_idx)}")

take = min(N_QUESTIONS, len(fact_idx))
records = []
for idx in range(take):                      # idx = factoid ordinal = id
    row = ds[fact_idx[idx]]
    records.append({
        "id":            f"factoid-{idx:05d}",
        "question":      str(row.get("Question", "")).strip(),
        "choices":       None,
        "answer":        _gold(row.get("Answer")),
        "question_type": "factoid",
        "image":         _save_image(row.get("image"), IMAGES_DIR / f"{idx:05d}.png"),
    })

DATASET_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2))
print(f"first {len(records)} factoid questions + images → {DATASET_JSON}")


In [ ]:
# ── sanity: unchanged RoSE config + stage resume checkpoints ─────
import rose_chartqapro as rose
assert (rose.M_PATHS, rose.K_DEMONSTRATIONS, rose.LAMBDA, rose.TEMPERATURE) \
       == (5, 3, 1.2, 0.8), "rose_chartqapro defaults drifted from the sweep config"
assert not rose.PAPER_FAITHFUL
rose.IMAGES_DIR = str(IMAGES_DIR)
print(f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  T={rose.TEMPERATURE}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

# The one embedder for everything: RoSE demo retrieval, the 7B pool,
# and Algorithm 2's decomposition vote.
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(rose.EMBED_MODEL)
print(f"embedder: {rose.EMBED_MODEL}")

import shutil
for alg, src in (RESUME_FROM or {}).items():
    if src:
        ckpt = RESULTS_DIR / f"decomp_alg{alg}_checkpoint.json"
        shutil.copy(src, ckpt)
        print(f"resume checkpoint staged for algorithm {alg} → {ckpt}")


In [ ]:
# ── run both algorithms (GPU) ────────────────────────────────────
# All three 4-bit VLMs are loaded ONCE and stay resident across both
# algorithms (interleave=True; ≈13 GB — needs the T4 x2). Questions run
# in dataset order with a checkpoint after EVERY question, so a partial
# run is an unbiased prefix and nothing is lost if the session dies.
# Each run_decomp call gets whatever is LEFT of the session budget:
# Algorithm 2 starts only after Algorithm 1 completes, and a budgeted
# stop between questions still publishes every output file.
from decompose.decomp_runner import load_all_models, run_decomp

loaded, revisions = load_all_models(QUANT)

all_results = {}
for alg in ALGORITHMS:
    remaining = None
    if SESSION_BUDGET_H is not None:
        remaining = SESSION_BUDGET_H * 3600 - (time.time() - T_START)
        if remaining <= 0:
            print(f"⏱  no session budget left — algorithm {alg} not started; "
                  "chain another version (RESUME_FROM) to continue")
            continue
    all_results[alg] = run_decomp(
        alg, DATASET_JSON, RESULTS_DIR / f"decomp_alg{alg}.json",
        n=N_QUESTIONS, quant=QUANT,
        embedder=embedder, loaded=loaded, revisions=revisions,
        limit=2 if DRY_RUN else None,
        stop_after_s=remaining, check_access=False)   # access verified above


In [ ]:
# ── report ───────────────────────────────────────────────────────
# The runner already wrote, per algorithm: decomp_alg<A>.json (+ _checkpoint),
# decomp_alg<A>.csv (per question, rescored with scoring.py),
# decomp_alg<A>_stages.csv (per-stage latency/accuracy) and
# decomp_alg<A>_meta.json (revisions, versions, algorithm config).
import json

print(f"{'arm':<12} {'answered':>9} {'accuracy':>9} {'params/q':>9} "
      f"{'latency/q':>10} {'complete':>9}")
for alg in ALGORITHMS:
    meta_path = RESULTS_DIR / f"decomp_alg{alg}_meta.json"
    if not meta_path.exists():
        print(f"decomp-alg{alg}  (not run this session)")
        continue
    m = json.loads(meta_path.read_text())
    lat = f"{m['mean_latency_s']:8.1f}s" if m.get("mean_latency_s") else "        —"
    print(f"decomp-alg{alg} {m['n']:>9} {m['accuracy_rescored']:8.1f}% "
          f"{m['params_b_used']:8.2f}B {lat} {str(m['complete']):>9}")
    if m.get("vote_rules"):
        rules = "  ".join(f"{k}: {v}" for k, v in sorted(m["vote_rules"].items()))
        print(f"             vote rules — {rules}")

print("\ndownload from the Output tab, per algorithm: decomp_alg<A>.json, "
      "decomp_alg<A>.csv, decomp_alg<A>_stages.csv, decomp_alg<A>_meta.json, "
      "decomp_alg<A>_checkpoint.json")
print("then, with the routed CSV and the cached 7B results in place:  "
      "cd chartqapro && python results/compare_arms.py")


## After the session

**Headless chaining (Save & Run All):** with `SESSION_BUDGET_H` set, each
committed version runs up to the budget, stops cleanly between questions,
and PUBLISHES its outputs. To chain sessions: Add Input → Your Work →
this notebook (attaches the latest version's output), set
`RESUME_FROM = {1: "/kaggle/input/<slug>/results/decompose/decomp_alg1_checkpoint.json", 2: …}`
for whichever algorithms are unfinished, then Save & Run All again.
Answered questions are skipped and errored rows retried. Repeat until
both meta files say `"complete": true`. Re-pin the input to the newest
version before each re-run.

1. Download from the Output pane, per algorithm: `decomp_alg<A>.json`,
   `decomp_alg<A>.csv`, `decomp_alg<A>_stages.csv`,
   `decomp_alg<A>_meta.json` (and the `_checkpoint.json` if unfinished).
2. Drop the finished files into `chartqapro/results/decompose/` in the
   repo. Each meta file records the resolved model revisions, installed
   library versions and the full algorithm config, so a finished run is
   self-describing.
3. The cross-arm table (accuracy, mean params/q, mean latency/q, and the
   Alg 1 sub-answer-vs-final confusion counts) comes from
   `python results/compare_arms.py` run inside `chartqapro/` — it
   rescores every arm with `scoring.py` and restricts them to their
   common question window.
4. Quota bookkeeping: ~29–33 GPU-h for Algorithm 1 and ~21 GPU-h for
   Algorithm 2 at N=250 on a T4 x2 — plan roughly five 10 h-budgeted
   sessions, run in series.
